# Mood-Aware Recommendation Engine
This notebook builds the recommendation engine on top of the Qdrant vector index created in `04_build_vector_index.ipynb`. 

### Key design decision: intent detection
A user saying **"I am sad"** and **"I want a sad book"** have opposite needs:

| Query | Intent | What we want |
|-------|--------|--------------|
| *"I am sad"* | Emotional state | Comfort, warmth, escapism — books that lift |
| *"I want a sad book"* | Content request | Heartbreaking, heavy, cathartic books |

We handle this with an **intent classifier** that runs before the similarity search, then rewrites the query accordingly.

### Vector dimension note

Notebook 04 stored **387-dimensional** vectors: 384 (sentence embedding) + 3 (scaled metadata: `description_word_count`, `num_tags`, `average_rating`). Query vectors must match this dimension. For the metadata dimensions we append the **training set mean** of each feature — a neutral, average-book assumption.


In [1]:
import pandas as pd
import numpy as np
import joblib
import ast
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, Range

print('Libraries loaded.')

Libraries loaded.


## Load Artefacts

In [2]:
# Load books dataframe (needed for mean metadata @computation)
df = pd.read_csv('../data/processed/books_with_features.csv')
df['tags_cleaned'] = df['tags_cleaned'].apply(ast.literal_eval)

# Load the same scaler fitted in notebook 04
scaler = joblib.load('../artifacts/metadata_scaler.pkl')

# Load sentence transformer — must be the same model used in 03_feature_engineering
print('Loading sentence transformer...')
model = SentenceTransformer('all-MiniLM-L6-v2')

# Connect to Qdrant
client = QdrantClient(host='localhost', port=6333)
COLLECTION = 'book_recommender'

print(f'Connected to Qdrant. Collection: {COLLECTION}')
print(f'Books in dataframe: {len(df)}')

Loading sentence transformer...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Connected to Qdrant. Collection: book_recommender
Books in dataframe: 5045


## Compute mean metadata vector
When embedding a user query, we have no real values for `description_word_count`, `num_tags`, or `average_rating` — the query is not a book. We append the **scaled mean** of each feature, which places the query at the centroid of the metadata space — a neutral assumption that avoids biasing results toward short or long, tagged or untagged books.

In [3]:
METADATA_COLS = ['description_word_count', 'num_tags', 'average_rating']

# Compute mean metadata row and scale it using the saved scaler
mean_metadata_raw = df[METADATA_COLS].mean().values.reshape(1, -1)
mean_metadata_scaled = scaler.transform(mean_metadata_raw)  # shape: (1, 3)

print('Mean metadata (raw):   ', mean_metadata_raw.round(2))
print('Mean metadata (scaled):', mean_metadata_scaled.round(4))
print('\nThese 3 values will be appended to every query embedding.')

Mean metadata (raw):    [[144.26  46.03   3.89]]
Mean metadata (scaled): [[0. 0. 0.]]

These 3 values will be appended to every query embedding.


## Intent detection
Zero-shot classification using anchor sentences. We define representative sentences for each intent class, embedd them, take the mean vector per class, and classify new queries by closest centroid. 

In [4]:
INTENT_ANCHORS = {
    'SEEK_CONTENT': [
        'I want to read a book that makes me feel sad',
        'recommend me a dark and heavy book',
        'I am looking for a thrilling story',
        'suggest a funny book for me to read',
        'I want a book about adventure',
    ],
    'EMOTIONAL_STATE': [
        'I am feeling sad right now',
        'I feel anxious and stressed',
        'I am happy and energetic today',
        'I feel lonely',
        'I am bored and restless',
    ]
}

# Pre-compute class centroids (mean embedding per intent class)
anchor_embeddings = {}
for intent, sentences in INTENT_ANCHORS.items():
    vecs = model.encode(sentences, convert_to_numpy=True)
    anchor_embeddings[intent] = vecs.mean(axis=0, keepdims=True)  # shape: (1, 384)

print('Intent anchors computed.')


def detect_intent(query: str, threshold: float = 0.05) -> tuple[str, dict]:
    """
    Classify query as SEEK_CONTENT or EMOTIONAL_STATE.

    Uses cosine similarity against pre-computed anchor centroids.
    If the margin between the two classes is below `threshold`,
    defaults to SEEK_CONTENT (safer assumption).
    """
    query_vec = model.encode([query], convert_to_numpy=True)  # (1, 384)

    scores = {
        intent: cosine_similarity(query_vec, centroid)[0][0]
        for intent, centroid in anchor_embeddings.items()
    }

    best = max(scores, key=scores.get)
    other = [k for k in scores if k != best][0]
    margin = scores[best] - scores[other]

    intent = best if margin >= threshold else 'SEEK_CONTENT'
    return intent, scores


# Sanity check
test_queries = [
    'I am sad',
    'I want a sad book',
    'I feel anxious',
    'recommend me something dark',
]

print(f"\n{'Query':<45} {'Intent'}")
print('-' * 65)
for q in test_queries:
    intent, scores = detect_intent(q)
    print(f'{q:<45} {intent}')

Intent anchors computed.

Query                                         Intent
-----------------------------------------------------------------
I am sad                                      EMOTIONAL_STATE
I want a sad book                             SEEK_CONTENT
I feel anxious                                EMOTIONAL_STATE
recommend me something dark                   SEEK_CONTENT


## Mood inversion map

In [5]:
MOOD_INVERSION = [
    (
        ['sad', 'unhappy', 'miserable', 'depressed', 'down', 'low', 'heartbroken', 'grief'],
        'uplifting, warm, comforting story with hope and humour'
    ),
    (
        ['anxious', 'stressed', 'overwhelmed', 'worried', 'nervous', 'tense'],
        'calm, gentle, cosy story with a slow pace and reassuring tone'
    ),
    (
        ['bored', 'restless', 'uninspired', 'flat', 'stuck'],
        'gripping adventure with unexpected twists and high energy'
    ),
    (
        ['lonely', 'isolated', 'disconnected'],
        'warm story about friendship, community and human connection'
    ),
    (
        ['angry', 'frustrated', 'irritated', 'furious'],
        'absorbing plot-driven story that pulls you completely out of your head'
    ),
    (
        ['happy', 'excited', 'energetic', 'joyful', 'great'],
        'fun, witty, playful book to match good energy'
    ),
    (
        ['tired', 'exhausted', 'drained'],
        'light, easy-to-read, comforting story that does not demand too much'
    ),
]


def rewrite_emotional_query(query: str) -> tuple[str, str]:
    """
    Rewrite an EMOTIONAL_STATE query into a book-content search query.

    Returns
    -------
    rewritten_query : str  — used for semantic search
    explanation     : str  — shown to the user
    """
    query_lower = query.lower()
    for trigger_words, rewritten in MOOD_INVERSION:
        if any(word in query_lower for word in trigger_words):
            return rewritten, f'You seem to be feeling {trigger_words[0]} — recommending something to help with that.'
    # Fallback
    return f'comforting and engaging {query}', 'Recommending something that fits your mood.'


print('Mood inversion map ready.')

Mood inversion map ready.


## Build query vector

In [6]:
def build_query_vector(search_query: str) -> list[float]:
    """
    Embed a search query into a 387-dim vector matching the Qdrant collection.

    384 dims : sentence embedding from all-MiniLM-L6-v2
      3 dims : scaled mean metadata (neutral book assumption)
    """
    text_vec = model.encode([search_query], convert_to_numpy=True)  # (1, 384)
    combined = np.hstack([text_vec, mean_metadata_scaled])           # (1, 387)
    return combined[0].tolist()


# Verify dimension
test_vec = build_query_vector('a cosy autumn read')
print(f'Query vector dimension: {len(test_vec)}  (expected 387)')

Query vector dimension: 387  (expected 387)


## Core recommendation function

In [7]:
def recommend(
    query: str,
    top_k: int = 10,
    min_rating: float = 3.5,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Recommend books given a free-text mood query.

    Parameters
    ----------
    query      : Free-text user input, e.g. 'I am sad' or 'I want a dark thriller'
    top_k      : Number of recommendations to return
    min_rating : Minimum average_rating filter applied inside Qdrant
    verbose    : Print intent detection info

    Returns
    -------
    DataFrame of top_k recommended books with similarity scores
    """

    # --- Step 1: Intent detection ---
    intent, scores = detect_intent(query)

    # --- Step 2: Query rewriting ---
    if intent == 'EMOTIONAL_STATE':
        search_query, explanation = rewrite_emotional_query(query)
    else:
        search_query = query
        explanation = None

    if verbose:
        print(f'Query:        "{query}"')
        print(f'Intent:       {intent}')
        print(f'Search query: "{search_query}"')
        if explanation:
            print(f'Note:         {explanation}')
        print()

    # --- Step 3: Build 387-dim query vector ---
    query_vector = build_query_vector(search_query)

    # --- Step 4: Query Qdrant with optional rating filter ---
    rating_filter = Filter(
        must=[
            FieldCondition(
                key='average_rating',
                range=Range(gte=min_rating)
            )
        ]
    )

    hits = client.query_points(
    collection_name=COLLECTION,
    query=query_vector,
    query_filter=rating_filter,
    limit=top_k,
    with_payload=True
).points

    # --- Step 5: Format results ---
    rows = []
    for hit in hits:
        rows.append({
            'title':            hit.payload.get('title', 'Unknown'),
            'authors':          hit.payload.get('authors', ''),
            'average_rating':   hit.payload.get('average_rating', None),
            'moods':            ', '.join(hit.payload.get('moods', [])),
            'similarity_score': round(hit.score, 4),
        })

    return pd.DataFrame(rows)


print('recommend() defined.')

recommend() defined.


In [8]:
print('=' * 60)
print('QUERY 1: "I am sad"')
print('=' * 60)
results_1 = recommend('I am sad', top_k=5)
print(results_1[['title', 'authors', 'average_rating', 'moods', 'similarity_score']].to_string(index=False))

QUERY 1: "I am sad"
Query:        "I am sad"
Intent:       EMOTIONAL_STATE
Search query: "uplifting, warm, comforting story with hope and humour"
Note:         You seem to be feeling sad — recommending something to help with that.

                                                  title              authors  average_rating                                      moods  similarity_score
                           The Walls Came Tumbling Down           ['757047']            3.84                               intellectual            0.4677
                                        Star of the Sea            ['34713']            3.97 dark, emotional, adventurous, intellectual            0.4597
                                        Star Of The Sea            ['34713']            3.97 dark, emotional, adventurous, intellectual            0.4552
A HEAVENLY CHRISTMAS (Heavenly Christmas series Book 1)          ['1806225']            3.80                                                       0.450

In [9]:
print('=' * 60)
print('QUERY 2: "I want a sad book"')
print('=' * 60)
results_2 = recommend('I want a sad book', top_k=5)
print(results_2[['title', 'authors', 'average_rating', 'moods', 'similarity_score']].to_string(index=False))

QUERY 2: "I want a sad book"
Query:        "I want a sad book"
Intent:       SEEK_CONTENT
Search query: "I want a sad book"

                                                   title   authors  average_rating                                         moods  similarity_score
The Ersatz Elevator (A Series of Unfortunate Events, #6) ['36746']            4.01 dark, light, emotional, adventurous, escapist            0.4163
                                         Star of the Sea ['34713']            3.97    dark, emotional, adventurous, intellectual            0.4067
                                          Fahrenheit 451  ['1630']            3.97                        intellectual, escapist            0.3979
                                  The Accidental Tourist   ['457']            3.90                light, emotional, intellectual            0.3817
                                         Star Of The Sea ['34713']            3.97    dark, emotional, adventurous, intellectual            

In [10]:
# How different are the two lists?
titles_1 = set(results_1['title'])
titles_2 = set(results_2['title'])
overlap = titles_1 & titles_2

print(f'Overlap between results: {len(overlap)}/5 books')
if overlap:
    print(f'Shared titles: {overlap}')
    print('\nConsider tuning the intent anchor sentences if overlap is high.')
else:
    print('No overlap — intent detection is working correctly.')

Overlap between results: 2/5 books
Shared titles: {'Star Of The Sea', 'Star of the Sea'}

Consider tuning the intent anchor sentences if overlap is high.


In [11]:
test_queries = [
    'I feel anxious and overwhelmed',
    'I want something dark and unsettling',
    'I am bored, give me something gripping',
    'I want a cosy autumn read',
    'I feel lonely',
    'recommend me something that will make me cry',
]

for q in test_queries:
    print('\n' + '=' * 60)
    results = recommend(q, top_k=3)
    print(results[['title', 'authors', 'similarity_score']].to_string(index=False))

Query:        "I feel anxious and overwhelmed"
Intent:       EMOTIONAL_STATE
Search query: "calm, gentle, cosy story with a slow pace and reassuring tone"
Note:         You seem to be feeling anxious — recommending something to help with that.

                  title              authors  similarity_score
Lips Touch: Three Times ['324620', '428160']            0.4440
 The Accidental Tourist              ['457']            0.4199
                   Emma             ['1265']            0.4151

Query:        "I want something dark and unsettling"
Intent:       SEEK_CONTENT
Search query: "I want something dark and unsettling"

                               title              authors  similarity_score
        Certain Dark Things: Stories          ['7821729']            0.4119
             Lips Touch: Three Times ['324620', '428160']            0.3478
A Journey to the Center of the Earth           ['696805']            0.3443

Query:        "I am bored, give me something gripping"
Intent: 

In [12]:
def diagnose_intent(query: str):
    """Print detailed intent classification breakdown for a query."""
    intent, scores = detect_intent(query)
    margin = abs(scores['SEEK_CONTENT'] - scores['EMOTIONAL_STATE'])

    print(f'Query:          "{query}"')
    print(f'SEEK_CONTENT:   {scores["SEEK_CONTENT"]:.4f}')
    print(f'EMOTIONAL_STATE:{scores["EMOTIONAL_STATE"]:.4f}')
    print(f'Margin:         {margin:.4f}  (threshold=0.05)')
    print(f'Result:         {intent}')
    print()


# Test any queries you are unsure about here
diagnose_intent('I am sad')
diagnose_intent('I want a sad book')
diagnose_intent('something melancholic')

Query:          "I am sad"
SEEK_CONTENT:   0.2118
EMOTIONAL_STATE:0.5561
Margin:         0.3443  (threshold=0.05)
Result:         EMOTIONAL_STATE

Query:          "I want a sad book"
SEEK_CONTENT:   0.7383
EMOTIONAL_STATE:0.3684
Margin:         0.3699  (threshold=0.05)
Result:         SEEK_CONTENT

Query:          "something melancholic"
SEEK_CONTENT:   0.3065
EMOTIONAL_STATE:0.3167
Margin:         0.0102  (threshold=0.05)
Result:         SEEK_CONTENT

